# 🔤 Classic NLP: From Raw Text to Features
### Tokenization, Normalization, BoW, N-grams, and TF-IDF

**Key Insight:** In classic NLP, most wins come from *representation choices*, not fancy modeling!

---
## Setup: Install and Import Libraries

In [88]:
# Install required libraries (run once)
# !pip install nltk scikit-learn pandas

In [89]:
import pandas as pd
import numpy as np
import re
import nltk
from collections import Counter

# Download NLTK resources (run once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

print("✅ All libraries ready!")

✅ All libraries ready!


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/shivam13juna/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


---
## Load the IMDB Dataset

In [90]:
# Load the dataset
# Source: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

df = pd.read_csv('imdb_dataset.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Dataset shape: (50000, 2)
Columns: ['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [91]:
# Check the sentiment distribution
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [92]:
# Let's look at a few sample reviews
for i, row in df.head(3).iterrows():
    print(f"\n{'='*60}")
    print(f"Sentiment: {row['sentiment'].upper()}")
    print(f"Review: {row['review'][:200]}...")


Sentiment: POSITIVE
Review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Sentiment: POSITIVE
Review: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece...

Sentiment: POSITIVE
Review: I thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. The plot is simplistic, but the dialogue is wi...


---
# PART 1: Text → Tokens
## Tokenization & Normalization Choices

---
## 1.1 What is a Token?

A **token** is a single unit of text. Think of it as "words" but more flexible.

In [93]:
# Simple example: What is a token?
sentence = "I love NLP!"

# METHOD 1: Simplest approach - split by spaces
tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
print("Number of tokens:", len(tokens_simple))

Split by space: ['I', 'love', 'NLP!']
Number of tokens: 3


In [94]:
# But wait... what about punctuation?
sentence = "Hello, world! How are you?"

tokens_simple = sentence.split()
print("Split by space:", tokens_simple)
# Problem: "Hello," and "you?" have punctuation attached!

Split by space: ['Hello,', 'world!', 'How', 'are', 'you?']


---
## 1.2 Tokenization Methods Compared

In [95]:
from nltk.tokenize import word_tokenize, TreebankWordTokenizer

# A complex sentence to tokenize
text = "I can't believe http://wow.com is FREE!!! It's amazing @user 🔥"

print("Original text:")
print(text)
print("\n" + "="*60)

Original text:
I can't believe http://wow.com is FREE!!! It's amazing @user 🔥



In [96]:
# Method 1: Simple whitespace split
tokens_whitespace = text.split()
print("\n1️⃣ Whitespace split:")
print(tokens_whitespace)
print(f"   → {len(tokens_whitespace)} tokens")


1️⃣ Whitespace split:
['I', "can't", 'believe', 'http://wow.com', 'is', 'FREE!!!', "It's", 'amazing', '@user', '🔥']
   → 10 tokens


In [97]:
# Method 2: NLTK's word_tokenize (smarter!)
tokens_nltk = word_tokenize(text)
print("\n2️⃣ NLTK word_tokenize:")
print(tokens_nltk)
print(f"   → {len(tokens_nltk)} tokens")
print("   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)")


2️⃣ NLTK word_tokenize:
['I', 'ca', "n't", 'believe', 'http', ':', '//wow.com', 'is', 'FREE', '!', '!', '!', 'It', "'s", 'amazing', '@', 'user', '🔥']
   → 18 tokens
   → Notice: 'can't' → 'ca' + 'n't' (handles contractions!)


In [98]:
# Method 3: Regex-based tokenization (custom control)
tokens_regex = re.findall(r'\b\w+\b', text)
print("\n3️⃣ Regex (\\b\\w+\\b):")
print(tokens_regex)
print(f"   → {len(tokens_regex)} tokens")
print("   → Notice: Punctuation and URLs handled differently!")


3️⃣ Regex (\b\w+\b):
['I', 'can', 't', 'believe', 'http', 'wow', 'com', 'is', 'FREE', 'It', 's', 'amazing', 'user']
   → 13 tokens
   → Notice: Punctuation and URLs handled differently!


In [99]:
# COMPARISON TABLE
print("\n" + "="*60)
print("COMPARISON: Same text, different tokenizers")
print("="*60)
comparison = pd.DataFrame({
    'Method': ['Whitespace', 'NLTK', 'Regex'],
    'Token Count': [len(tokens_whitespace), len(tokens_nltk), len(tokens_regex)],
    'Handles Contractions': ['❌', '✅', '❌'],
    'Handles Punctuation': ['❌', '✅', '✅'],
})
comparison


COMPARISON: Same text, different tokenizers


,Method,Token Count,Handles Contractions,Handles Punctuation
0,Whitespace,10,❌,❌
1,NLTK,18,✅,✅
2,Regex,13,❌,✅


---
## 1.3 Lowercasing: When It Helps vs Hurts

In [100]:
# WHEN LOWERCASING HELPS
text1 = "I love Apple products. APPLE makes great devices."

tokens_original = word_tokenize(text1)
tokens_lower = word_tokenize(text1.lower())

print("Original tokens:", tokens_original)
print("Lowercased tokens:", tokens_lower)
print("\n✅ BENEFIT: 'Apple' and 'APPLE' are now the same token!")

Original tokens: ['I', 'love', 'Apple', 'products', '.', 'APPLE', 'makes', 'great', 'devices', '.']
Lowercased tokens: ['i', 'love', 'apple', 'products', '.', 'apple', 'makes', 'great', 'devices', '.']

✅ BENEFIT: 'Apple' and 'APPLE' are now the same token!


In [101]:
# WHEN LOWERCASING HURTS
text2 = "I work in IT. It is interesting. US policy affects us."

print("Original:", text2)
print("Lowercased:", text2.lower())

print("\n❌ PROBLEM:")
print("   'IT' (technology) → 'it' (pronoun) - meaning lost!")
print("   'US' (country) → 'us' (pronoun) - meaning lost!")

Original: I work in IT. It is interesting. US policy affects us.
Lowercased: i work in it. it is interesting. us policy affects us.

❌ PROBLEM:
   'IT' (technology) → 'it' (pronoun) - meaning lost!
   'US' (country) → 'us' (pronoun) - meaning lost!


In [102]:
# Let's see the impact on our IMDB dataset
sample_review = df['review'].iloc[0]

tokens_original = word_tokenize(sample_review)
tokens_lower = word_tokenize(sample_review.lower())

# Count unique tokens
print(f"Original unique tokens: {len(set(tokens_original))}")
print(f"Lowercased unique tokens: {len(set(tokens_lower))}")
print(f"\n✅ Vocabulary reduced by {len(set(tokens_original)) - len(set(tokens_lower))} tokens!")

Original unique tokens: 210
Lowercased unique tokens: 202

✅ Vocabulary reduced by 8 tokens!


---
## 1.4 Handling Special Tokens

In [103]:
# Special tokens: URLs, emails, mentions, numbers
text = "Check out https://example.com or email me@test.com! Price: $199.99 @username #hashtag"

print("Original:", text)
print("\nProblem: These special patterns can cause noise!")

Original: Check out https://example.com or email me@test.com! Price: $199.99 @username #hashtag

Problem: These special patterns can cause noise!


In [ ]:
# Function to replace special tokens with placeholders
def normalize_special_tokens(text):
    """Replace URLs, emails, mentions, numbers with placeholders"""
    
    # Replace URLs
    # Pattern: r'https?://\S+|www\.\S+'
    # - https? : matches 'http' or 'https' (? makes 's' optional)
    # - :// : matches the literal '://' in URLs
    # - \S+ : matches one or more non-whitespace characters (the rest of URL)
    # - | : OR operator
    # - www\. : matches literal 'www.' (dot is escaped with \)
    # - \S+ : matches rest of URL after www.
    text = re.sub(r'https?://\S+|www\.\S+', '<URL>', text)
    
    # Replace emails
    # Pattern: r'\S+@\S+\.\S+'
    # - \S+ : matches one or more non-whitespace chars (username part)
    # - @ : matches literal '@' symbol
    # - \S+ : matches domain name (e.g., 'gmail')
    # - \. : matches literal dot (escaped)
    # - \S+ : matches domain extension (e.g., 'com')
    text = re.sub(r'\S+@\S+\.\S+', '<EMAIL>', text)
    
    # Replace @mentions
    # Pattern: r'@\w+'
    # - @ : matches literal '@' symbol
    # - \w+ : matches one or more word characters (letters, digits, underscore)
    # - Example: '@user123' → '<MENTION>'
    text = re.sub(r'@\w+', '<MENTION>', text)
    
    # Replace #hashtags
    # Pattern: r'#\w+'
    # - # : matches literal '#' symbol
    # - \w+ : matches one or more word characters after the hash
    # - Example: '#python' → '<HASHTAG>'
    text = re.sub(r'#\w+', '<HASHTAG>', text)
    
    # Replace money amounts
    # Pattern: r'\$[\d,]+\.?\d*'
    # - \$ : matches literal dollar sign (escaped)
    # - [\d,]+ : matches one or more digits or commas (e.g., '1,234')
    # - \.? : matches optional decimal point (? makes it 0 or 1 time)
    # - \d* : matches zero or more digits after decimal (* means 0 or more)
    # - Examples: '$199', '$1,234.56', '$50.5' → '<MONEY>'
    text = re.sub(r'\$[\d,]+\.?\d*', '<MONEY>', text)
    
    # Replace numbers
    # Pattern: r'\b\d+\b'
    # - \b : word boundary (ensures we match whole numbers, not parts)
    # - \d+ : matches one or more digits
    # - \b : word boundary at the end
    # - Example: '123' in 'I have 123 apples' → '<NUM>'
    # - Won't match '123' inside 'abc123def' due to word boundaries
    text = re.sub(r'\b\d+\b', '<NUM>', text)
    
    return text

# Apply to our example
normalized = normalize_special_tokens(text)
print("\nNormalized:")
print(normalized)



Normalized:
Check out <URL> or email <EMAIL> Price: <MONEY> <MENTION> <HASHTAG>


In [105]:
# Apply to IMDB data - note the <br /> tags in the data
sample = df['review'].iloc[1]
print("Before:", sample[:100])

# Also clean HTML tags
def clean_html(text):
    return re.sub(r'<[^>]+>', ' ', text)

cleaned = clean_html(sample)
print("\nAfter HTML cleaning:", cleaned[:100])

Before: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-B

After HTML cleaning: A wonderful little production.   The filming technique is very unassuming- very old-time-BBC fashion


---
## 1.5 ⚠️ DANGER: Stopwords and Negation

In [106]:
from nltk.corpus import stopwords

# Get English stopwords
stop_words = set(stopwords.words('english'))

print(f"Number of stopwords: {len(stop_words)}")
print(f"\nSome examples: {list(stop_words)[:20]}")

Number of stopwords: 198

Some examples: ['himself', "it's", 'have', 'your', 'up', 'ours', 'are', "don't", 'isn', "doesn't", 'its', 'myself', "they'd", 'when', "wasn't", "weren't", 'and', 'of', 'above', "he'll"]


In [107]:
# ⚠️ THE DANGER: Negation words are often in stopword lists!
negation_words = ['not', 'no', 'never', 'neither', 'nor', "n't", 'cannot', "don't", "won't", "didn't"]

print("Checking if negation words are in stopwords list:")
for word in negation_words:
    status = "⚠️ IN STOPWORDS!" if word in stop_words else "✅ Not in stopwords"
    print(f"  '{word}': {status}")

Checking if negation words are in stopwords list:
  'not': ⚠️ IN STOPWORDS!
  'no': ⚠️ IN STOPWORDS!
  'never': ✅ Not in stopwords
  'neither': ✅ Not in stopwords
  'nor': ⚠️ IN STOPWORDS!
  'n't': ✅ Not in stopwords
  'cannot': ✅ Not in stopwords
  'don't': ⚠️ IN STOPWORDS!
  'won't': ⚠️ IN STOPWORDS!
  'didn't': ⚠️ IN STOPWORDS!


In [108]:
# DEMONSTRATION: Why this matters
sentence = "I do not like this movie"

tokens = word_tokenize(sentence.lower())
print("Original tokens:", tokens)

# Remove stopwords (DANGEROUS!)
tokens_no_stop = [t for t in tokens if t not in stop_words]
print("After stopword removal:", tokens_no_stop)

print("\n🚨 MEANING REVERSED!")
print("   'I do not like this movie' → 'like movie'")
print("   Negative sentiment → looks positive!")

Original tokens: ['i', 'do', 'not', 'like', 'this', 'movie']
After stopword removal: ['like', 'movie']

🚨 MEANING REVERSED!
   'I do not like this movie' → 'like movie'
   Negative sentiment → looks positive!


In [109]:
# SOLUTION: Create a custom stopword list that PRESERVES negation
negation_words_to_keep = {'not', 'no', 'never', 'neither', 'nor', "n't", 'cannot', 
                          "don't", "won't", "didn't", "wasn't", "isn't", 
                          "aren't", "haven't", "hasn't", "hadn't", "couldn't",
                          "shouldn't", "wouldn't", "but"}

# Safe stopwords = original stopwords MINUS negation words
safe_stopwords = stop_words - negation_words_to_keep

# Now remove stopwords safely
tokens_safe = [t for t in tokens if t not in safe_stopwords]
print("With SAFE stopword removal:", tokens_safe)
print("\n✅ 'not' is preserved! Meaning intact.")

With SAFE stopword removal: ['not', 'like', 'movie']

✅ 'not' is preserved! Meaning intact.


In [110]:
# More examples
test_sentences = [
    "This is not good",
    "I would never recommend this",
    "There is no way this works",
    "I didn't like the ending"
]

print("Comparison: Standard vs Safe Stopword Removal\n")
for sent in test_sentences:
    tokens = word_tokenize(sent.lower())
    standard = [t for t in tokens if t not in stop_words]
    safe = [t for t in tokens if t not in safe_stopwords]
    
    print(f"Original: {sent}")
    print(f"  ❌ Standard: {standard}")
    print(f"  ✅ Safe:     {safe}")
    print()

Comparison: Standard vs Safe Stopword Removal

Original: This is not good
  ❌ Standard: ['good']
  ✅ Safe:     ['not', 'good']

Original: I would never recommend this
  ❌ Standard: ['would', 'never', 'recommend']
  ✅ Safe:     ['would', 'never', 'recommend']

Original: There is no way this works
  ❌ Standard: ['way', 'works']
  ✅ Safe:     ['no', 'way', 'works']

Original: I didn't like the ending
  ❌ Standard: ["n't", 'like', 'ending']
  ✅ Safe:     ["n't", 'like', 'ending']



---
## 1.6 Punctuation: Keep or Remove?

In [111]:
# Punctuation can carry MEANING!
examples = [
    "This is great!!!",      # Strong positive
    "This is great.",        # Neutral positive
    "This is great...",      # Uncertain/hesitant
    "This is great?",        # Sarcastic/questioning
]

print("Same words, different punctuation = different meaning!\n")
for ex in examples:
    print(f"  {ex}")

Same words, different punctuation = different meaning!

  This is great!!!
  This is great.
  This is great...
  This is great?


In [112]:
# The classic example: Punctuation saves lives!
print("Let's eat grandma!!")
print("Let's eat, grandma!!")
print("\n🍽️ Grandma's life depends on that comma!")

Let's eat grandma!!
Let's eat, grandma!!

🍽️ Grandma's life depends on that comma!


In [113]:
# Function to handle punctuation thoughtfully
import string

def remove_punctuation(text, keep_important=True):
    """Remove punctuation, optionally keeping important ones"""
    if keep_important:
        # Keep ! and ? as they carry sentiment
        punctuation = string.punctuation.replace('!', '').replace('?', '')
    else:
        punctuation = string.punctuation
    
    return text.translate(str.maketrans('', '', punctuation))

text = "Wow!!! This is amazing??? I can't believe it..."
print(f"Original: {text}")
print(f"Remove all: {remove_punctuation(text, keep_important=False)}")
print(f"Keep !?: {remove_punctuation(text, keep_important=True)}")

Original: Wow!!! This is amazing??? I can't believe it...
Remove all: Wow This is amazing I cant believe it
Keep !?: Wow!!! This is amazing??? I cant believe it


---
## 1.7 Stemming vs Lemmatization


### Very common inflectional endings

* **`-s`, `-es`**
  Examples: `cats → cat`, `caresses → caress`, `ponies → poni` (Porter often uses `i`)
* **`-ed`, `-ing`**
  Examples: `agreed → agree` (via `eed → ee`/`e` rules), `playing → play`, `hopping → hop`
* **`-y → -i`** (when there’s a vowel earlier)
  Example: `happy → happi`

### Common derivational endings (often more “word-formation” than grammar)

These show up in later steps and can be **removed** or **rewritten**:

* **`-ation/-ization/-izer`**
  Examples: `organization → organ`, `realization → realiz → real`, `computerizer → computerize → comput`
* **`-ational/-tional`**
  Examples: `relational → relat`, `conditional → condit`
* **`-ness`, `-ful`**
  Examples: `happiness → happi`, `usefulness → use`
* **`-al/-ical/-ic`**
  Examples: `logical → logic → log`, `political → polit`
* **`-ive/-ative`**
  Examples: `talkative → talk`, `decisive → decis`
* **`-ment`, `-ement`**
  Examples: `adjustment → adjust`, `replacement → replac`
* **`-able/-ible`**
  Examples: `readable → read`, `sensible → sens`
* **`-ity/-aliti/-iviti/-biliti`**
  Examples: `sensitivity → sensit`, `capability → capabl`
* **`-ous/-ousness`**
  Examples: `famous → fam`, `generousness → gener`
* **`-ism`**
  Example: `capitalism → capit`


In [114]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer

# Initialize stemmers and lemmatizer
porter = PorterStemmer()
snowball = SnowballStemmer('english')
lemmatizer = WordNetLemmatizer()

print("Stemmer vs Lemmatizer: What's the difference?\n")

Stemmer vs Lemmatizer: What's the difference?



In [115]:
# Test words
words = ['running', 'runs', 'ran', 'studies', 'studying', 'better', 'cats', 'wolves', 'feet']

print(f"{'Word':<12} {'Porter Stem':<15} {'Snowball Stem':<15} {'Lemma':<12}")
print("-" * 55)

for word in words:
    stem_porter = porter.stem(word)
    stem_snowball = snowball.stem(word)
    lemma = lemmatizer.lemmatize(word)
    print(f"{word:<12} {stem_porter:<15} {stem_snowball:<15} {lemma:<12}")

Word         Porter Stem     Snowball Stem   Lemma       
-------------------------------------------------------
running      run             run             running     
runs         run             run             run         
ran          ran             ran             ran         
studies      studi           studi           study       
studying     studi           studi           studying    
better       better          better          better      
cats         cat             cat             cat         
wolves       wolv            wolv            wolf        
feet         feet            feet            foot        


In [116]:
# KEY INSIGHT: Stemming can produce NON-WORDS!
print("\n🔍 Key Observation:")
print(f"   'studies' stemmed → '{porter.stem('studies')}' (not a real word!)")
print(f"   'studies' lemmatized → '{lemmatizer.lemmatize('studies', pos='v')}' (real word!)")

print("\n📊 Tradeoff:")
print("   Stemming: Fast, rule-based, but can produce non-words")
print("   Lemmatization: Slower, uses dictionary, always produces real words")


🔍 Key Observation:
   'studies' stemmed → 'studi' (not a real word!)
   'studies' lemmatized → 'study' (real word!)

📊 Tradeoff:
   Stemming: Fast, rule-based, but can produce non-words
   Lemmatization: Slower, uses dictionary, always produces real words


In [117]:
# Lemmatization needs POS tag for best results
word = 'better'

print(f"Lemmatizing '{word}':")
print(f"  Default (noun): {lemmatizer.lemmatize(word)}")
print(f"  As adjective: {lemmatizer.lemmatize(word, pos='a')}")

# With correct POS, lemmatizer returns 'good'!

Lemmatizing 'better':
  Default (noun): better
  As adjective: good


In [118]:
# Apply to a sentence
sentence = "The striped bats are hanging on their feet for best results"
tokens = word_tokenize(sentence.lower())

stemmed = [porter.stem(t) for t in tokens]
lemmatized = [lemmatizer.lemmatize(t) for t in tokens]

print(f"Original: {tokens}")
print(f"Stemmed:  {stemmed}")
print(f"Lemmatized: {lemmatized}")

Original: ['the', 'striped', 'bats', 'are', 'hanging', 'on', 'their', 'feet', 'for', 'best', 'results']
Stemmed:  ['the', 'stripe', 'bat', 'are', 'hang', 'on', 'their', 'feet', 'for', 'best', 'result']
Lemmatized: ['the', 'striped', 'bat', 'are', 'hanging', 'on', 'their', 'foot', 'for', 'best', 'result']


In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')

# Test spaCy lemmatizer
spacy_test_words = ['good', 'better', 'best', 'running', 'runs', 'ran', 'studies', 'studying', 'cats', 'wolves', 'feet']
print("spaCy Lemmatization:")
for word in spacy_test_words:
    doc = nlp(word)
    lemma_spacy = doc[0].lemma_
    print(f"Word: {word:10} | spaCy Lemma: {lemma_spacy:10}")

print("\n" + "="*60 + "\n")

---
## 1.8 Putting It All Together: Full Preprocessing Pipeline

In [ ]:
def preprocess_text(text, 
                    lowercase=True,
                    remove_html=True,
                    remove_urls=True,
                    remove_stopwords=True,
                    preserve_negation=True,
                    stem=False,
                    lemmatize=False):
    """
    Complete text preprocessing pipeline.
    
    Parameters:
    -----------
    text : str - Input text
    lowercase : bool - Convert to lowercase
    remove_html : bool - Remove HTML tags
    remove_urls : bool - Remove URLs
    remove_stopwords : bool - Remove stopwords
    preserve_negation : bool - Keep negation words when removing stopwords
    stem : bool - Apply Porter stemming
    lemmatize : bool - Apply lemmatization
    
    Returns:
    --------
    list of tokens
    """
    
    # Step 1: Remove HTML
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)
    
    # Step 2: Remove URLs
    if remove_urls:
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Step 3: Lowercase
    if lowercase:
        text = text.lower()
    
    # Step 4: Tokenize
    tokens = word_tokenize(text)
    
    # Step 5: Remove stopwords (safely!)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        if preserve_negation:
            negation = {'not', 'no', 'never', 'neither', 'nor', "n't", 
                       'cannot', "don't", "won't", "didn't", "wasn't", 
                       "isn't", "aren't", "haven't", "hasn't", "hadn't", 
                       "couldn't", "shouldn't", "wouldn't", "but"}
            stop_words = stop_words - negation
        tokens = [t for t in tokens if t not in stop_words]
    
    # Step 6: Remove non-alphabetic tokens
    tokens = [t for t in tokens if t.isalpha() or t == "n't"]
    
    # Step 7: Stem or Lemmatize
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(t) for t in tokens]
    elif lemmatize:
        doc = nlp(' '.join(tokens))
        tokens = [token.lemma_ for token in doc]
    
    return tokens

In [120]:
# Test the pipeline
sample_text = df['review'].iloc[0]
print("Original text:")
print(sample_text[:200], "...")
print("\n" + "="*60)

# Process with different settings
tokens_basic = preprocess_text(sample_text)
tokens_stemmed = preprocess_text(sample_text, stem=True)
tokens_lemma = preprocess_text(sample_text, lemmatize=True)

print(f"\nBasic preprocessing: {len(tokens_basic)} tokens")
print(tokens_basic[:15], "...")

print(f"\nWith stemming: {len(tokens_stemmed)} tokens")
print(tokens_stemmed[:15], "...")

print(f"\nWith lemmatization: {len(tokens_lemma)} tokens")
print(tokens_lemma[:15], "...")

Original text:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo ...


Basic preprocessing: 173 tokens
['one', 'reviewers', 'mentioned', 'watching', 'oz', 'episode', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality'] ...

With stemming: 173 tokens
['one', 'review', 'mention', 'watch', 'oz', 'episod', 'hook', 'right', 'exactli', 'happen', 'first', 'thing', 'struck', 'oz', 'brutal'] ...

With lemmatization: 173 tokens
['one', 'reviewer', 'mentioned', 'watching', 'oz', 'episode', 'hooked', 'right', 'exactly', 'happened', 'first', 'thing', 'struck', 'oz', 'brutality'] ...


In [121]:
# Apply to entire dataset
print("Processing entire dataset...")
df['tokens'] = df['review'].apply(lambda x: preprocess_text(x, lemmatize=True))
print("✅ Done!")

# Show result
df[['review', 'tokens', 'sentiment']].head()

Processing entire dataset...
✅ Done!


,review,tokens,sentiment
0,One of the other reviewers has mentioned that ...,"[one, reviewer, mentioned, watching, oz, episo...",positive
1,A wonderful little production. <br /><br />The...,"[wonderful, little, production, filming, techn...",positive
2,I thought this was a wonderful way to spend ti...,"[thought, wonderful, way, spend, time, hot, su...",positive
3,Basically there's a family where a little boy ...,"[basically, family, little, boy, jake, think, ...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, love, time, money, visually, ...",positive


---
# CHECKPOINT: Part 1 Complete!

### What we covered:
- ✅ Tokenization splits text into units
- ✅ Lowercasing: task-dependent decision
- ✅ Handle special tokens (URLs, HTML, numbers)
- ✅ **Preserve negation words** (not, no, never)
- ✅ Stemming/lemmatization reduces vocabulary

### What's next:
- Convert tokens into numeric vectors
- Three approaches: **Counts → Context → Weighting**

---
---
# PART 2: Tokens → Vectors
## Three Approaches: Counts, Context, Weighting

---
## 2A: Bag-of-Words (Counts)
### The Simplest Numeric Representation

In [122]:
# Core idea: Count how many times each word appears
# Ignore word order completely!

sentence1 = "the cat sat on the mat"
sentence2 = "the dog sat on the cat"

# Manual BoW
def manual_bow(text):
    words = text.lower().split()
    return Counter(words)

bow1 = manual_bow(sentence1)
bow2 = manual_bow(sentence2)

print(f"Sentence 1: '{sentence1}'")
print(f"BoW: {dict(bow1)}")
print(f"\nSentence 2: '{sentence2}'")
print(f"BoW: {dict(bow2)}")

Sentence 1: 'the cat sat on the mat'
BoW: {'the': 2, 'cat': 1, 'sat': 1, 'on': 1, 'mat': 1}

Sentence 2: 'the dog sat on the cat'
BoW: {'the': 2, 'dog': 1, 'sat': 1, 'on': 1, 'cat': 1}


In [ ]:
# Now let's create a vector representation
# Step 1: Build vocabulary from both sentences
all_words = list(set(bow1.keys()) | set(bow2.keys()))
all_words.sort()  # Sort for consistency
print("Vocabulary:", all_words)
print(f"Vocabulary size: {len(all_words)}")

Vocabulary: ['cat', 'dog', 'mat', 'on', 'sat', 'the']
Vocabulary size: 6


In [124]:
# Step 2: Create vectors
def to_vector(bow, vocabulary):
    return [bow.get(word, 0) for word in vocabulary]

vec1 = to_vector(bow1, all_words)
vec2 = to_vector(bow2, all_words)

print("Vector representation:")
print(f"Vocabulary: {all_words}")
print(f"Sentence 1: {vec1}")
print(f"Sentence 2: {vec2}")

Vector representation:
Vocabulary: ['cat', 'dog', 'mat', 'on', 'sat', 'the']
Sentence 1: [1, 0, 1, 1, 1, 2]
Sentence 2: [1, 1, 0, 1, 1, 2]


In [125]:
# Visual representation
bow_df = pd.DataFrame(
    [vec1, vec2],
    columns=all_words,
    index=['"the cat sat on the mat"', '"the dog sat on the cat"']
)
print("Document-Term Matrix:")
bow_df

Document-Term Matrix:


,cat,dog,mat,on,sat,the
"""the cat sat on the mat""",1,0,1,1,1,2
"""the dog sat on the cat""",1,1,0,1,1,2


In [126]:
# Using sklearn's CountVectorizer (the professional way)
from sklearn.feature_extraction.text import CountVectorizer

# Create vectorizer
count_vectorizer = CountVectorizer()

# Sample documents
documents = [
    "the cat sat on the mat",
    "the dog sat on the cat",
    "the cat and dog are friends"
]

# Fit and transform
bow_matrix = count_vectorizer.fit_transform(documents)

# Get feature names (vocabulary)
vocab = count_vectorizer.get_feature_names_out()
print("Vocabulary:", vocab)

Vocabulary: ['and' 'are' 'cat' 'dog' 'friends' 'mat' 'on' 'sat' 'the']


In [127]:
# View as DataFrame
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=vocab,
    index=[f"Doc {i+1}" for i in range(len(documents))]
)
bow_df

,and,are,cat,dog,friends,mat,on,sat,the
Doc 1,0,0,1,0,0,1,1,1,2
Doc 2,0,0,1,1,0,0,1,1,2
Doc 3,1,1,1,1,1,0,0,0,1


In [128]:
# ⚠️ BOW FAILURE MODE: Order is lost!
print("\n🚨 BoW PROBLEM: Word order is completely ignored!")
print()

sentences = [
    "not good",
    "good not"
]

vec = CountVectorizer()
matrix = vec.fit_transform(sentences)

print(f"Vocabulary: {vec.get_feature_names_out()}")
print(f"'not good' → {matrix.toarray()[0]}")
print(f"'good not' → {matrix.toarray()[1]}")
print("\n❌ Same vector! BoW can't distinguish these!")


🚨 BoW PROBLEM: Word order is completely ignored!

Vocabulary: ['good' 'not']
'not good' → [1 1]
'good not' → [1 1]

❌ Same vector! BoW can't distinguish these!


In [129]:
# More failure examples
failure_examples = [
    ("I love to hate it", "I hate to love it"),
    ("dog bites man", "man bites dog"),
    ("this is good", "is this good"),
]

print("BoW treats these pairs as IDENTICAL:\n")
for s1, s2 in failure_examples:
    vec = CountVectorizer()
    matrix = vec.fit_transform([s1, s2])
    identical = np.array_equal(matrix.toarray()[0], matrix.toarray()[1])
    print(f"  '{s1}'")
    print(f"  '{s2}'")
    print(f"  → Same vector: {identical}")
    print()

BoW treats these pairs as IDENTICAL:

  'I love to hate it'
  'I hate to love it'
  → Same vector: True

  'dog bites man'
  'man bites dog'
  → Same vector: True

  'this is good'
  'is this good'
  → Same vector: True



---
## 2A.1: Apply BoW to IMDB Dataset

In [130]:
# Apply BoW to our IMDB data
from sklearn.feature_extraction.text import CountVectorizer

# Clean the text first
def clean_text(text):
    # Remove HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    # Lowercase
    text = text.lower()
    return text

df['clean_text'] = df['review'].apply(clean_text)

# Create BoW with max 1000 features (for speed)
vectorizer = CountVectorizer(max_features=1000, stop_words='english')
X_bow = vectorizer.fit_transform(df['clean_text'])

print(f"BoW matrix shape: {X_bow.shape}")
print(f"  → {X_bow.shape[0]} documents")
print(f"  → {X_bow.shape[1]} unique words (features)")

BoW matrix shape: (50000, 1000)
  → 50000 documents
  → 1000 unique words (features)


In [131]:
# Look at the most common words
word_counts = X_bow.sum(axis=0).A1  # Sum across all documents
word_freq = list(zip(vectorizer.get_feature_names_out(), word_counts))
word_freq.sort(key=lambda x: x[1], reverse=True)

print("Top 20 most frequent words in corpus:")
for word, count in word_freq[:20]:
    print(f"  {word}: {int(count)}")

Top 20 most frequent words in corpus:
  movie: 87970
  film: 79705
  like: 40172
  just: 35183
  good: 29753
  time: 25109
  story: 23119
  really: 23094
  bad: 18473
  people: 18188
  great: 18144
  don: 17623
  make: 15898
  way: 15645
  movies: 15309
  characters: 14456
  think: 14337
  watch: 13946
  character: 13905
  films: 13755


In [132]:
df['sentiment'] == 'positive'

0         True
1         True
2         True
3        False
4         True
         ...  
49995     True
49996    False
49997    False
49998    False
49999    False
Name: sentiment, Length: 50000, dtype: bool

In [133]:
# Compare positive vs negative reviews
positive_mask = df['sentiment'] == 'positive'
negative_mask = df['sentiment'] == 'negative'

# Convert pandas Series to numpy arrays for sparse matrix indexing
# This prevents the 'Series' object has no attribute 'nonzero' error
positive_counts = X_bow[positive_mask.values].sum(axis=0).A1
negative_counts = X_bow[negative_mask.values].sum(axis=0).A1

# Create comparison DataFrame
comparison = pd.DataFrame({
    'word': vectorizer.get_feature_names_out(),
    'positive_count': positive_counts,
    'negative_count': negative_counts,
})

# Calculate ratio (positive / negative)
comparison['ratio'] = (comparison['positive_count'] + 1) / (comparison['negative_count'] + 1)

print("\n🟢 Words more common in POSITIVE reviews:")
print(comparison.nlargest(10, 'ratio')[['word', 'positive_count', 'negative_count', 'ratio']])

print("\n🔴 Words more common in NEGATIVE reviews:")
print(comparison.nsmallest(10, 'ratio')[['word', 'positive_count', 'negative_count', 'ratio']])



🟢 Words more common in POSITIVE reviews:
          word  positive_count  negative_count     ratio
847     superb            1116             184  6.037838
973  wonderful            2668             551  4.835145
279  excellent            3359             745  4.504021
303  fantastic            1228             291  4.208904
32     amazing            2004             516  3.878143
665   powerful             960             275  3.481884
98   brilliant            1874             540  3.465804
460    journey             702             205  3.412621
629    perfect            2434             720  3.377254
308   favorite            1843             548  3.358834

🔴 Words more common in NEGATIVE reviews:
          word  positive_count  negative_count     ratio
949      waste             178            2611  0.068530
982      worst             446            4888  0.091430
59       awful             304            3143  0.097010
655     poorly             134            1258  0.107228
650 

---
# CHECKPOINT: Part 2A Complete!

### What BoW can express:
- ✅ Word presence/frequency
- ✅ Topic-related keywords
- ✅ Spam indicators (FREE, $$$, etc.)

### What BoW can't express:
- ❌ Word order
- ❌ Negation ("not good" ≈ "good not")
- ❌ Phrases ("New York" is 2 separate words)

### Solution: N-grams!

---
## 2B: N-grams (Local Context)
### Capturing Word Order

In [134]:
# What is an N-gram?
# N consecutive words treated as a single feature

from nltk import ngrams

sentence = "I love natural language processing"
tokens = sentence.lower().split()

print(f"Sentence: '{sentence}'")
print(f"Tokens: {tokens}")
print()

# Generate n-grams
unigrams = list(ngrams(tokens, 1))
bigrams = list(ngrams(tokens, 2))
trigrams = list(ngrams(tokens, 3))

print(f"Unigrams (n=1): {unigrams}")
print(f"Bigrams  (n=2): {bigrams}")
print(f"Trigrams (n=3): {trigrams}")

Sentence: 'I love natural language processing'
Tokens: ['i', 'love', 'natural', 'language', 'processing']

Unigrams (n=1): [('i',), ('love',), ('natural',), ('language',), ('processing',)]
Bigrams  (n=2): [('i', 'love'), ('love', 'natural'), ('natural', 'language'), ('language', 'processing')]
Trigrams (n=3): [('i', 'love', 'natural'), ('love', 'natural', 'language'), ('natural', 'language', 'processing')]


In [135]:
# N-grams SOLVE the negation problem!
print("🔥 N-grams solve the 'not good' problem!\n")

sentences = [
    "this movie is not good",
    "this movie is very good"
]

for sent in sentences:
    tokens = sent.lower().split()
    bi = list(ngrams(tokens, 2))
    print(f"'{sent}'")
    print(f"  Bigrams: {bi}")
    print()

print("✅ 'not good' and 'very good' are now DIFFERENT features!")

🔥 N-grams solve the 'not good' problem!

'this movie is not good'
  Bigrams: [('this', 'movie'), ('movie', 'is'), ('is', 'not'), ('not', 'good')]

'this movie is very good'
  Bigrams: [('this', 'movie'), ('movie', 'is'), ('is', 'very'), ('very', 'good')]

✅ 'not good' and 'very good' are now DIFFERENT features!


In [136]:
# Using sklearn's CountVectorizer with n-grams
from sklearn.feature_extraction.text import CountVectorizer

# Unigrams only (default BoW)
vec_unigram = CountVectorizer(ngram_range=(1, 1))

# Bigrams only
vec_bigram = CountVectorizer(ngram_range=(2, 2))

# Unigrams + Bigrams (common practice!)
vec_combined = CountVectorizer(ngram_range=(1, 2))

sentences = [
    "not good movie",
    "very good movie",
    "not bad actually"
]

print("Unigrams only:")
X1 = vec_unigram.fit_transform(sentences)
print(vec_unigram.get_feature_names_out())

print("\nBigrams only:")
X2 = vec_bigram.fit_transform(sentences)
print(vec_bigram.get_feature_names_out())

print("\nUnigrams + Bigrams:")
X3 = vec_combined.fit_transform(sentences)
print(vec_combined.get_feature_names_out())

Unigrams only:
['actually' 'bad' 'good' 'movie' 'not' 'very']

Bigrams only:
['bad actually' 'good movie' 'not bad' 'not good' 'very good']

Unigrams + Bigrams:
['actually' 'bad' 'bad actually' 'good' 'good movie' 'movie' 'not'
 'not bad' 'not good' 'very' 'very good']


In [137]:
# Compare vectors for "not good" vs "very good"
test_sentences = ["not good", "very good"]

print("With UNIGRAMS only:")
vec1 = CountVectorizer(ngram_range=(1, 1))
X1 = vec1.fit_transform(test_sentences)
df1 = pd.DataFrame(X1.toarray(), columns=vec1.get_feature_names_out(), index=test_sentences)
print(df1)
print("❌ Can't distinguish these!\n")

print("With UNIGRAMS + BIGRAMS:")
vec2 = CountVectorizer(ngram_range=(1, 2))
X2 = vec2.fit_transform(test_sentences)
df2 = pd.DataFrame(X2.toarray(), columns=vec2.get_feature_names_out(), index=test_sentences)
print(df2)
print("✅ Now they're different!")

With UNIGRAMS only:
           good  not  very
not good      1    1     0
very good     1    0     1
❌ Can't distinguish these!

With UNIGRAMS + BIGRAMS:
           good  not  not good  very  very good
not good      1    1         1     0          0
very good     1    0         0     1          1
✅ Now they're different!


In [138]:
# Real-world negation examples with LONGER sentences
negation_sentences = [
    "This movie was not good at all",        # Negative - 'not good'
    "The film was not bad actually",         # Positive - 'not bad' (double negative!)
    "I found it very good and entertaining", # Strong positive - 'very good'
    "The acting was hardly good enough",     # Negative - 'hardly good' (hedging)
    "This was never good from the start",    # Strong negative - 'never good'
]

print("Bigrams capture OVERLAPPING context throughout the sentence:\n")
for sent in negation_sentences:
    tokens = sent.lower().split()
    bi = list(ngrams(tokens, 2))
    
    # Find the key bigram with modifier + 'good'/'bad'
    key_bigrams = [str(b) for b in bi if 'good' in b or 'bad' in b]
    
    print(f"'{sent}'")
    print(f"  All bigrams: {bi}")
    print(f"  Key bigram: {key_bigrams}")
    print()

print("✅ With longer sentences, you get MULTIPLE overlapping bigrams!")
print("   Each captures LOCAL context around important words.")


Bigrams capture OVERLAPPING context throughout the sentence:

'This movie was not good at all'
  All bigrams: [('this', 'movie'), ('movie', 'was'), ('was', 'not'), ('not', 'good'), ('good', 'at'), ('at', 'all')]
  Key bigram: ["('not', 'good')", "('good', 'at')"]

'The film was not bad actually'
  All bigrams: [('the', 'film'), ('film', 'was'), ('was', 'not'), ('not', 'bad'), ('bad', 'actually')]
  Key bigram: ["('not', 'bad')", "('bad', 'actually')"]

'I found it very good and entertaining'
  All bigrams: [('i', 'found'), ('found', 'it'), ('it', 'very'), ('very', 'good'), ('good', 'and'), ('and', 'entertaining')]
  Key bigram: ["('very', 'good')", "('good', 'and')"]

'The acting was hardly good enough'
  All bigrams: [('the', 'acting'), ('acting', 'was'), ('was', 'hardly'), ('hardly', 'good'), ('good', 'enough')]
  Key bigram: ["('hardly', 'good')", "('good', 'enough')"]

'This was never good from the start'
  All bigrams: [('this', 'was'), ('was', 'never'), ('never', 'good'), ('good'

In [139]:
# Apply to IMDB dataset
print("Creating n-gram features for IMDB dataset...\n")

# Unigrams only
vec_uni = CountVectorizer(max_features=1000, stop_words='english', ngram_range=(1,1))
X_uni = vec_uni.fit_transform(df['clean_text'])

# Unigrams + Bigrams
vec_combo = CountVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))
X_combo = vec_combo.fit_transform(df['clean_text'])

print(f"Unigrams only: {X_uni.shape[1]} features")
print(f"Uni + Bigrams: {X_combo.shape[1]} features")
print(f"\n⚠️ Notice: Adding bigrams almost DOUBLES the feature space!")

Creating n-gram features for IMDB dataset...

Unigrams only: 1000 features
Uni + Bigrams: 2000 features

⚠️ Notice: Adding bigrams almost DOUBLES the feature space!


In [140]:
# Find the most common bigrams
bigram_vec = CountVectorizer(ngram_range=(2,2), max_features=100, stop_words='english')
X_bi = bigram_vec.fit_transform(df['clean_text'])

bigram_counts = X_bi.sum(axis=0).A1
bigram_freq = list(zip(bigram_vec.get_feature_names_out(), bigram_counts))
bigram_freq.sort(key=lambda x: x[1], reverse=True)

print("Top 15 most common bigrams:")
for bi, count in bigram_freq[:15]:
    print(f"  '{bi}': {int(count)}")

Top 15 most common bigrams:
  've seen': 4168
  'special effects': 2249
  'don know': 2202
  'low budget': 1824
  'looks like': 1678
  'year old': 1598
  'movie just': 1538
  'waste time': 1526
  'good movie': 1515
  'sci fi': 1393
  'watch movie': 1377
  'new york': 1316
  'look like': 1312
  'don think': 1305
  'years ago': 1256


In [141]:
# Find bigrams that differ between positive and negative reviews
positive_bi = X_bi[positive_mask.values].sum(axis=0).A1
negative_bi = X_bi[negative_mask.values].sum(axis=0).A1

bi_comparison = pd.DataFrame({
    'bigram': bigram_vec.get_feature_names_out(),
    'positive': positive_bi,
    'negative': negative_bi
})
bi_comparison['ratio'] = (bi_comparison['positive'] + 1) / (bi_comparison['negative'] + 1)

print("🟢 Bigrams more common in POSITIVE reviews:")
print(bi_comparison.nlargest(8, 'ratio')[['bigram', 'positive', 'negative']])

print("\n🔴 Bigrams more common in NEGATIVE reviews:")
print(bi_comparison.nsmallest(8, 'ratio')[['bigram', 'positive', 'negative']])

🟢 Bigrams more common in POSITIVE reviews:
              bigram  positive  negative
33  highly recommend       503        53
0              10 10       734        82
29        great film       576       148
30       great movie       723       225
58       movie great       463       174
91         world war       403       159
99         young man       367       157
79   supporting cast       426       189

🔴 Bigrams more common in NEGATIVE reviews:
         bigram  positive  negative
92  worst movie        23       743
12    don waste        21       508
85   waste time        73      1453
1    bad acting        60       694
71   really bad        92       719
55    movie bad        85       654
4     bad movie       151       926
40   just plain       142       532


---
# CHECKPOINT: Part 2B Complete!

### What N-grams add:
- ✅ Captures local word order
- ✅ Handles negation beautifully ("not good" is ONE feature)
- ✅ Finds meaningful phrases

### Tradeoffs:
- ⚠️ More features = more sparsity
- ⚠️ Many n-grams appear only once
- ⚠️ Need more data to estimate reliably

### Still a problem:
- Common words like "the", "movie" dominate the counts!
- Solution: TF-IDF weighting

---
## 2C: TF-IDF (Weighting)
### Fixing the Dominance of Common Words

In [142]:
# The Problem: Common words dominate
print("The problem with raw counts:")
print("  'the' appears 1000 times - but tells us NOTHING")
print("  'excellent' appears 5 times - but is VERY informative!")
print("\nWe want: Distinctive words to matter MORE")

The problem with raw counts:
  'the' appears 1000 times - but tells us NOTHING
  'excellent' appears 5 times - but is VERY informative!

We want: Distinctive words to matter MORE


In [143]:
# TF-IDF = Term Frequency × Inverse Document Frequency
import math

# Simple demonstration
documents = [
    "the movie was great and fun",
    "the movie was terrible",
    "the film was excellent and great"
]

# Total documents
N = len(documents)

# Count how many documents contain each word
word_doc_count = Counter()
for doc in documents:
    unique_words = set(doc.lower().split())
    for word in unique_words:
        word_doc_count[word] += 1

print("Document frequency (how many docs contain each word):")
for word, count in sorted(word_doc_count.items()):
    print(f"  '{word}': {count}/{N} documents")

Document frequency (how many docs contain each word):
  'and': 2/3 documents
  'excellent': 1/3 documents
  'film': 1/3 documents
  'fun': 1/3 documents
  'great': 2/3 documents
  'movie': 2/3 documents
  'terrible': 1/3 documents
  'the': 3/3 documents
  'was': 3/3 documents


In [144]:
# Calculate IDF manually
print("\nIDF calculation: log(N / doc_frequency)")
print(f"N = {N} (total documents)\n")

for word in ['the', 'movie', 'was', 'great', 'excellent', 'terrible']:
    df_word = word_doc_count.get(word, 0)
    if df_word > 0:
        idf = math.log(N / df_word) + 1  # +1 is smoothing
        print(f"  '{word}': appears in {df_word} docs → IDF = log({N}/{df_word})+1 = {idf:.2f}")
    else:
        print(f"  '{word}': not found")

print("\n💡 Key insight: Words in ALL docs get LOW IDF!")
print("   Words in FEW docs get HIGH IDF!")


IDF calculation: log(N / doc_frequency)
N = 3 (total documents)

  'the': appears in 3 docs → IDF = log(3/3)+1 = 1.00
  'movie': appears in 2 docs → IDF = log(3/2)+1 = 1.41
  'was': appears in 3 docs → IDF = log(3/3)+1 = 1.00
  'great': appears in 2 docs → IDF = log(3/2)+1 = 1.41
  'excellent': appears in 1 docs → IDF = log(3/1)+1 = 2.10
  'terrible': appears in 1 docs → IDF = log(3/1)+1 = 2.10

💡 Key insight: Words in ALL docs get LOW IDF!
   Words in FEW docs get HIGH IDF!


In [145]:
# Using sklearn's TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

documents = [
    "the movie was great and fun",
    "the movie was terrible",
    "the film was excellent and great"
]

# Transform
X_tfidf = tfidf_vectorizer.fit_transform(documents)

# View as DataFrame
tfidf_df = pd.DataFrame(
    X_tfidf.toarray().round(2),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("TF-IDF Matrix (values are weights, not raw counts):")
tfidf_df

TF-IDF Matrix (values are weights, not raw counts):


,and,excellent,film,fun,great,movie,terrible,the,was
Doc 1,0.41,0.00,0.00,0.54,0.41,0.41,0.00,0.32,0.32
Doc 2,0.00,0.00,0.00,0.00,0.00,0.50,0.66,0.39,0.39
Doc 3,0.39,0.51,0.51,0.00,0.39,0.00,0.00,0.30,0.30


In [146]:
# Compare with raw counts
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(documents)

count_df = pd.DataFrame(
    X_count.toarray(),
    columns=count_vec.get_feature_names_out(),
    index=[f"Doc {i+1}" for i in range(len(documents))]
)

print("Raw Counts:")
print(count_df)

print("\nTF-IDF Weights:")
print(tfidf_df.round(2))

print("\n💡 Notice:")
print("  - 'the', 'was' have LOWER TF-IDF than raw counts suggest")
print("  - 'excellent', 'terrible' have HIGHER relative weight")

Raw Counts:
       and  excellent  film  fun  great  movie  terrible  the  was
Doc 1    1          0     0    1      1      1         0    1    1
Doc 2    0          0     0    0      0      1         1    1    1
Doc 3    1          1     1    0      1      0         0    1    1

TF-IDF Weights:
        and  excellent  film   fun  great  movie  terrible   the   was
Doc 1  0.41       0.00  0.00  0.54   0.41   0.41      0.00  0.32  0.32
Doc 2  0.00       0.00  0.00  0.00   0.00   0.50      0.66  0.39  0.39
Doc 3  0.39       0.51  0.51  0.00   0.39   0.00      0.00  0.30  0.30

💡 Notice:
  - 'the', 'was' have LOWER TF-IDF than raw counts suggest
  - 'excellent', 'terrible' have HIGHER relative weight


In [147]:
# The three cases to understand
print("\n" + "="*60)
print("THREE CASES TO UNDERSTAND TF-IDF")
print("="*60)

print("""
CASE 1: High TF, High IDF → HIGH TF-IDF ✅
  Word appears FREQUENTLY in THIS document
  Word appears RARELY across corpus
  → This word is IMPORTANT for this document!
  Example: "excellent" in a positive review

CASE 2: High TF, Low IDF → LOW TF-IDF ❌
  Word appears FREQUENTLY in THIS document
  Word appears FREQUENTLY across corpus
  → Just a common word, not distinctive
  Example: "the", "movie"

CASE 3: Low TF, Low IDF → LOW TF-IDF ❌
  Word appears RARELY in THIS document
  Word appears FREQUENTLY across corpus
  → Common word, not emphasized here
  Example: "and" appearing once
""")


THREE CASES TO UNDERSTAND TF-IDF

CASE 1: High TF, High IDF → HIGH TF-IDF ✅
  Word appears FREQUENTLY in THIS document
  Word appears RARELY across corpus
  → This word is IMPORTANT for this document!
  Example: "excellent" in a positive review

CASE 2: High TF, Low IDF → LOW TF-IDF ❌
  Word appears FREQUENTLY in THIS document
  Word appears FREQUENTLY across corpus
  → Just a common word, not distinctive
  Example: "the", "movie"

CASE 3: Low TF, Low IDF → LOW TF-IDF ❌
  Word appears RARELY in THIS document
  Word appears FREQUENTLY across corpus
  → Common word, not emphasized here
  Example: "and" appearing once



In [148]:
# Apply TF-IDF to IMDB dataset
print("Applying TF-IDF to IMDB dataset...\n")

# TF-IDF with unigrams + bigrams
tfidf_vec = TfidfVectorizer(
    max_features=2000,
    stop_words='english',
    ngram_range=(1, 2)  # Unigrams and bigrams!
)

X_tfidf = tfidf_vec.fit_transform(df['clean_text'])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")
print(f"  → {X_tfidf.shape[0]} documents")
print(f"  → {X_tfidf.shape[1]} features (unigrams + bigrams)")

Applying TF-IDF to IMDB dataset...

TF-IDF matrix shape: (50000, 2000)
  → 50000 documents
  → 2000 features (unigrams + bigrams)


In [149]:
# Find words with highest IDF (most distinctive)
idf_scores = tfidf_vec.idf_
feature_names = tfidf_vec.get_feature_names_out()

idf_df = pd.DataFrame({
    'feature': feature_names,
    'idf': idf_scores
}).sort_values('idf', ascending=False)

print("Features with HIGHEST IDF (most distinctive/rare):")
print(idf_df.head(15))

print("\nFeatures with LOWEST IDF (most common):")
print(idf_df.tail(10))

Features with HIGHEST IDF (most distinctive/rare):
       feature       idf
837     hitler  6.351738
959     keaton  6.326737
155     batman  6.225087
64       alice  6.217679
1866  vampires  6.210326
713         fu  6.210326
63        alex  6.203027
1811      trek  6.106065
690   football  6.096213
399        dan  6.092951
1921     wayne  6.092951
1082     lynch  6.092951
82         ann  6.086457
76        andy  6.086457
936      jesus  6.073595

Features with LOWEST IDF (most common):
     feature       idf
771    great  2.380173
480      don  2.349173
1681   story  2.212697
1432  really  2.200798
1784    time  2.059431
756     good  1.967341
953     just  1.867902
1027    like  1.769357
647     film  1.588339
1176   movie  1.492122


In [150]:
# Find most important features for positive vs negative
# Calculate average TF-IDF per class

positive_tfidf = X_tfidf[positive_mask.values].mean(axis=0).A1
negative_tfidf = X_tfidf[negative_mask.values].mean(axis=0).A1

tfidf_comparison = pd.DataFrame({
    'feature': feature_names,
    'positive_avg': positive_tfidf,
    'negative_avg': negative_tfidf
})

tfidf_comparison['diff'] = tfidf_comparison['positive_avg'] - tfidf_comparison['negative_avg']

print("🟢 Features with HIGHEST TF-IDF in POSITIVE reviews:")
print(tfidf_comparison.nlargest(10, 'diff')[['feature', 'positive_avg', 'negative_avg', 'diff']])

print("\n🔴 Features with HIGHEST TF-IDF in NEGATIVE reviews:")
print(tfidf_comparison.nsmallest(10, 'diff')[['feature', 'positive_avg', 'negative_avg', 'diff']])

🟢 Features with HIGHEST TF-IDF in POSITIVE reviews:
        feature  positive_avg  negative_avg      diff
771       great      0.033125      0.012368  0.020757
1069       love      0.023957      0.011305  0.012652
171        best      0.022937      0.010650  0.012287
570   excellent      0.013535      0.002732  0.010803
1953  wonderful      0.011296      0.002237  0.009059
1023       life      0.020447      0.012511  0.007936
1072      loved      0.009972      0.002825  0.007147
159   beautiful      0.011083      0.004209  0.006874
1297    perfect      0.009457      0.002594  0.006863
71      amazing      0.008660      0.002110  0.006551

🔴 Features with HIGHEST TF-IDF in NEGATIVE reviews:
       feature  positive_avg  negative_avg      diff
139        bad      0.008882      0.037607 -0.028725
1968     worst      0.001386      0.018825 -0.017439
1176     movie      0.061326      0.078305 -0.016979
953       just      0.025720      0.038036 -0.012315
135      awful      0.001117      0.

---
## Final Comparison: Counts vs N-grams vs TF-IDF

In [151]:
# Compare all three representations for a single document
sample_doc = "This movie was not good at all. The acting was terrible and the plot made no sense."

# BoW (counts)
cv = CountVectorizer()
X_cv = cv.fit_transform([sample_doc])

# BoW with bigrams
cv_bi = CountVectorizer(ngram_range=(1,2))
X_bi = cv_bi.fit_transform([sample_doc])

# TF-IDF
tv = TfidfVectorizer(ngram_range=(1,2))
X_tv = tv.fit_transform([sample_doc])

print(f"Sample: '{sample_doc}'")
print(f"\nBoW (unigrams): {X_cv.shape[1]} features")
print(f"BoW + bigrams: {X_bi.shape[1]} features")
print(f"TF-IDF + bigrams: {X_tv.shape[1]} features")

Sample: 'This movie was not good at all. The acting was terrible and the plot made no sense.'

BoW (unigrams): 15 features
BoW + bigrams: 31 features
TF-IDF + bigrams: 31 features


In [152]:
# Look at the actual features for bigrams
print("Bigram features for this document:")
print("\nBigrams present:")
for feat in cv_bi.get_feature_names_out():
    if ' ' in feat:  # Only show bigrams
        print(f"  '{feat}'")

print("\n💡 Key insight: 'not good' is now captured as a single feature!")

Bigram features for this document:

Bigrams present:
  'acting was'
  'all the'
  'and the'
  'at all'
  'good at'
  'made no'
  'movie was'
  'no sense'
  'not good'
  'plot made'
  'terrible and'
  'the acting'
  'the plot'
  'this movie'
  'was not'
  'was terrible'

💡 Key insight: 'not good' is now captured as a single feature!


In [153]:
# Summary comparison table
print("\n" + "="*70)
print("COMPARISON: Which Representation to Use?")
print("="*70)

comparison_table = pd.DataFrame({
    'Representation': ['BoW (Counts)', 'N-grams', 'TF-IDF'],
    'Word Order': ['❌ Lost', '✅ Local', '✅ Local'],
    'Negation': ['❌ Bad', '✅ Good', '✅ Good'],
    'Common Words': ['❌ Dominate', '❌ Dominate', '✅ Downweighted'],
    'Sparsity': ['Low', 'High', 'High'],
    'Speed': ['Fast', 'Medium', 'Medium'],
    'Best For': ['Baseline', 'Sentiment', 'Classification']
})
comparison_table


COMPARISON: Which Representation to Use?


,Representation,Word Order,Negation,Common Words,Sparsity,Speed,Best For
0,BoW (Counts),❌ Lost,❌ Bad,❌ Dominate,Low,Fast,Baseline
1,N-grams,✅ Local,✅ Good,❌ Dominate,High,Medium,Sentiment
2,TF-IDF,✅ Local,✅ Good,✅ Downweighted,High,Medium,Classification


---
# 🎓 FINAL SUMMARY: Your NLP Preprocessing Checklist

1. **Normalize** URLs, numbers, mentions → placeholders
2. **Preserve negation words** (check stopword list!)
3. Start with **BoW** (Part 2A)
4. Add **bigrams** if negation matters (Part 2B)
5. Try **TF-IDF** if common words dominate (Part 2C)
6. **Always inspect** misclassified examples

---

## 🔑 Key Insight
### In classic NLP, representation is everything!

The choice of preprocessing and feature representation often matters MORE than the choice of model.

---
# 📚 Bonus: Quick Reference Functions

In [154]:
# Complete preprocessing pipeline - ready to use!

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import re

def create_nlp_features(texts, method='tfidf', ngram_range=(1,2), max_features=5000):
    """
    Create features from text using classic NLP methods.
    
    Parameters:
    -----------
    texts : list of str - Input texts
    method : str - 'bow' or 'tfidf'
    ngram_range : tuple - (min_n, max_n) for n-grams
    max_features : int - Maximum vocabulary size
    
    Returns:
    --------
    X : sparse matrix of features
    vectorizer : fitted vectorizer object
    """
    # Clean texts
    cleaned = []
    for text in texts:
        # Remove HTML
        text = re.sub(r'<[^>]+>', ' ', text)
        # Lowercase
        text = text.lower()
        cleaned.append(text)
    
    # Create vectorizer
    if method == 'bow':
        vectorizer = CountVectorizer(
            ngram_range=ngram_range,
            max_features=max_features,
            stop_words='english'
        )
    else:  # tfidf
        vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            max_features=max_features,
            stop_words='english'
        )
    
    # Fit and transform
    X = vectorizer.fit_transform(cleaned)
    
    return X, vectorizer

# Usage example
X, vec = create_nlp_features(df['review'].tolist(), method='tfidf')
print(f"Feature matrix: {X.shape}")
print(f"Sample features: {vec.get_feature_names_out()[:10]}")

Feature matrix: (50000, 5000)
Sample features: ['00' '000' '10' '10 10' '10 minutes' '10 stars' '10 years' '100' '11'
 '12']


---
# 🤖 Classification: Predicting Sentiment with BoW vs TF-IDF

So far we've built feature representations (BoW, N-grams, TF-IDF) but haven't actually used them to **predict** anything. Let's now train simple ML models on these features and compare which representation works best for sentiment classification on the IMDB dataset.

### What we'll do:
1. **Train/Test split** the IMDB reviews
2. Build **4 feature sets**: BoW (unigrams), BoW (uni+bigrams), TF-IDF (unigrams), TF-IDF (uni+bigrams)
3. Train two classic models on each: **Logistic Regression** and **Multinomial Naive Bayes**
4. **Compare accuracy** in a single table

### Why these models?
- **Logistic Regression** → strong linear baseline, handles sparse high-dim features well
- **Multinomial Naive Bayes** → classic text-classification workhorse, very fast

In [ ]:
# Step 1: Imports and Train/Test Split
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# Encode labels: positive -> 1, negative -> 0
y = (df['sentiment'] == 'positive').astype(int)

# Use the cleaned text we built earlier
X_text = df['clean_text']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train_text)}  |  Test size: {len(X_test_text)}")
print(f"Train positive ratio: {y_train.mean():.2f}")
print(f"Test  positive ratio: {y_test.mean():.2f}")

In [ ]:
# Step 2: Build the 4 feature representations
# IMPORTANT: fit ONLY on training text, then transform test text
# (otherwise we leak test info into our vocabulary / IDF values)

vectorizers = {
    'BoW (unigrams)':       CountVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 1)),
    'BoW (uni+bigrams)':    CountVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2)),
    'TF-IDF (unigrams)':    TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 1)),
    'TF-IDF (uni+bigrams)': TfidfVectorizer(max_features=5000, stop_words='english', ngram_range=(1, 2)),
}

features = {}
for name, vec in vectorizers.items():
    Xtr = vec.fit_transform(X_train_text)
    Xte = vec.transform(X_test_text)
    features[name] = (Xtr, Xte)
    print(f"{name:<25} → train shape: {Xtr.shape}, test shape: {Xte.shape}")

In [ ]:
# Step 3: Train Logistic Regression and Naive Bayes on each representation
results = []

for feat_name, (Xtr, Xte) in features.items():
    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, n_jobs=-1)
    lr.fit(Xtr, y_train)
    lr_acc = accuracy_score(y_test, lr.predict(Xte))

    # Multinomial Naive Bayes
    nb = MultinomialNB()
    nb.fit(Xtr, y_train)
    nb_acc = accuracy_score(y_test, nb.predict(Xte))

    results.append({
        'Features': feat_name,
        'Logistic Regression': lr_acc,
        'Naive Bayes': nb_acc,
    })
    print(f"{feat_name:<25} | LR: {lr_acc:.4f} | NB: {nb_acc:.4f}")

In [ ]:
# Step 4: Comparison table
results_df = pd.DataFrame(results).set_index('Features')
results_df['Best'] = results_df.idxmax(axis=1)
results_df_display = results_df.copy()
for col in ['Logistic Regression', 'Naive Bayes']:
    results_df_display[col] = results_df_display[col].apply(lambda v: f"{v:.4f}")
results_df_display

In [ ]:
# Step 5: Detailed report for the best combination
best_feat = results_df[['Logistic Regression', 'Naive Bayes']].max(axis=1).idxmax()
best_model_name = results_df.loc[best_feat, 'Best']
best_acc = results_df.loc[best_feat, best_model_name]

print(f"🏆 Best combination: {best_feat}  +  {best_model_name}")
print(f"   Test accuracy: {best_acc:.4f}\n")

# Refit and show full classification report
Xtr, Xte = features[best_feat]
model = LogisticRegression(max_iter=1000, n_jobs=-1) if best_model_name == 'Logistic Regression' else MultinomialNB()
model.fit(Xtr, y_train)
print(classification_report(y_test, model.predict(Xte), target_names=['negative', 'positive']))

### 🔍 What to look for in the results

- **TF-IDF usually beats raw BoW** for Logistic Regression because down-weighting common words helps the linear decision boundary.
- **Naive Bayes often does surprisingly well on raw counts** — it was literally designed for count data, and TF-IDF's float values can hurt its probability estimates.
- **Bigrams help with negation** (`not good`, `never again`) — watch whether `(uni+bigrams)` rows beat their unigram counterparts.
- The gap between best and worst is typically **only a few percentage points** — confirming the lecture's main point: *representation matters, but model choice on top of a decent representation matters less than you'd think.*

---

# Understanding Word Embeddings in Keras/TensorFlow

## The Embedding Layer - How It Works

An **Embedding Layer** is essentially a lookup table that maps integer indices (word IDs) to dense vectors of fixed size.

### The Process:
1. **Tokenization**: Convert words to unique integer indices
2. **Embedding Matrix Initialization**: Create a matrix of shape `(vocab_size, embedding_dim)` with random values
3. **Lookup**: For each word index, retrieve its corresponding row from the embedding matrix
4. **Training**: The embedding values are learned during training (they are weights!)

### Key Parameters:
- `input_dim`: Size of vocabulary (how many unique words)
- `output_dim`: Dimension of the embedding vectors (e.g., 50, 100, 300)
- `input_length`: Length of input sequences

In [1]:
# Step 1: Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding
from tensorflow.keras.models import Sequential

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


## Step 2: Create Dummy Sentences

Let's start with simple sentences to understand the process clearly.

In [2]:
# Our dummy sentences
dummy_sentences = [
    "I love machine learning",
    "I love deep learning",
    "machine learning is fun",
    "deep learning is amazing"
]

print("Our dummy sentences:")
for i, sent in enumerate(dummy_sentences):
    print(f"  {i+1}. {sent}")

Our dummy sentences:
  1. I love machine learning
  2. I love deep learning
  3. machine learning is fun
  4. deep learning is amazing


## Step 3: Tokenization - Assigning Word Indices

The **Tokenizer** creates a vocabulary and assigns a unique integer index to each word.

- Index 0 is typically reserved for padding
- Words are indexed by frequency (most common = lowest index)

In [3]:
# Create a tokenizer and fit on our sentences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(dummy_sentences)

# Get the word index (vocabulary)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1  # +1 because index 0 is reserved for padding

print("=" * 60)
print("WORD INDEX (Vocabulary)")
print("=" * 60)
print(f"\nTotal unique words: {len(word_index)}")
print(f"Vocabulary size (with padding): {vocab_size}")
print("\nWord → Index mapping:")
print("-" * 30)
for word, idx in sorted(word_index.items(), key=lambda x: x[1]):
    print(f"  '{word}' → {idx}")

WORD INDEX (Vocabulary)

Total unique words: 8
Vocabulary size (with padding): 9

Word → Index mapping:
------------------------------
  'learning' → 1
  'i' → 2
  'love' → 3
  'machine' → 4
  'deep' → 5
  'is' → 6
  'fun' → 7
  'amazing' → 8


## Step 4: Convert Sentences to Sequences of Indices

Each sentence becomes a sequence of integers (word indices).

In [4]:
# Convert sentences to sequences of integers
sequences = tokenizer.texts_to_sequences(dummy_sentences)

print("=" * 60)
print("SENTENCES → SEQUENCES")
print("=" * 60)
for sent, seq in zip(dummy_sentences, sequences):
    print(f"\nSentence: '{sent}'")
    print(f"Sequence: {seq}")
    # Show word-by-word mapping
    words = sent.lower().split()
    mapping = " → ".join([f"'{w}':{word_index[w]}" for w in words])
    print(f"Mapping:  {mapping}")

SENTENCES → SEQUENCES

Sentence: 'I love machine learning'
Sequence: [2, 3, 4, 1]
Mapping:  'i':2 → 'love':3 → 'machine':4 → 'learning':1

Sentence: 'I love deep learning'
Sequence: [2, 3, 5, 1]
Mapping:  'i':2 → 'love':3 → 'deep':5 → 'learning':1

Sentence: 'machine learning is fun'
Sequence: [4, 1, 6, 7]
Mapping:  'machine':4 → 'learning':1 → 'is':6 → 'fun':7

Sentence: 'deep learning is amazing'
Sequence: [5, 1, 6, 8]
Mapping:  'deep':5 → 'learning':1 → 'is':6 → 'amazing':8


## Step 5: Padding Sequences

Neural networks need fixed-length inputs. We pad shorter sequences with 0s.

In [5]:
# Pad sequences to same length
max_length = 5  # We'll pad all to length 5
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print("=" * 60)
print("PADDED SEQUENCES")
print("=" * 60)
print(f"\nMax length set to: {max_length}")
print(f"Padding value: 0 (represents nothing/padding)")
print("\nOriginal → Padded:")
print("-" * 50)
for sent, seq, padded in zip(dummy_sentences, sequences, padded_sequences):
    print(f"\n'{sent}'")
    print(f"  Original: {seq}")
    print(f"  Padded:   {list(padded)}")

PADDED SEQUENCES

Max length set to: 5
Padding value: 0 (represents nothing/padding)

Original → Padded:
--------------------------------------------------

'I love machine learning'
  Original: [2, 3, 4, 1]
  Padded:   [np.int32(2), np.int32(3), np.int32(4), np.int32(1), np.int32(0)]

'I love deep learning'
  Original: [2, 3, 5, 1]
  Padded:   [np.int32(2), np.int32(3), np.int32(5), np.int32(1), np.int32(0)]

'machine learning is fun'
  Original: [4, 1, 6, 7]
  Padded:   [np.int32(4), np.int32(1), np.int32(6), np.int32(7), np.int32(0)]

'deep learning is amazing'
  Original: [5, 1, 6, 8]
  Padded:   [np.int32(5), np.int32(1), np.int32(6), np.int32(8), np.int32(0)]


## Step 6: The Embedding Layer - The Magic Happens Here!

The Embedding layer:
1. Creates a matrix of shape `(vocab_size, embedding_dim)`
2. Initializes it with **random values**
3. Uses word indices to **look up** the corresponding embedding vector
4. These embeddings are **learnable weights** that get updated during training!

In [6]:
# Create an Embedding layer
embedding_dim = 4  # Each word will be represented by 4 numbers

# Create a simple model with just an embedding layer
embedding_layer = Embedding(
    input_dim=vocab_size,      # Size of vocabulary (10 words + padding)
    output_dim=embedding_dim,   # Dimension of embeddings (4-dimensional vectors)
    input_length=max_length     # Length of input sequences (5)
)

print("=" * 60)
print("EMBEDDING LAYER CONFIGURATION")
print("=" * 60)
print(f"\n• Vocabulary size (input_dim): {vocab_size}")
print(f"• Embedding dimension (output_dim): {embedding_dim}")
print(f"• Input sequence length: {max_length}")
print(f"\n• Embedding matrix shape: ({vocab_size}, {embedding_dim})")
print(f"  → {vocab_size} words × {embedding_dim} dimensions")
print(f"  → Total trainable parameters: {vocab_size * embedding_dim}")

EMBEDDING LAYER CONFIGURATION

• Vocabulary size (input_dim): 9
• Embedding dimension (output_dim): 4
• Input sequence length: 5

• Embedding matrix shape: (9, 4)
  → 9 words × 4 dimensions
  → Total trainable parameters: 36


/Users/shivam13juna/Documents/virtual_envs/dev/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


## Step 7: Visualize the Random Embedding Matrix

Let's see the actual embedding matrix - these are **random initial values** that will be learned during training.

In [7]:
# Build the layer by passing input
_ = embedding_layer(padded_sequences)

# Get the embedding matrix (weights)
embedding_matrix = embedding_layer.get_weights()[0]

print("=" * 60)
print("EMBEDDING MATRIX (Random Initialization)")
print("=" * 60)
print(f"\nShape: {embedding_matrix.shape}")
print(f"→ {embedding_matrix.shape[0]} rows (one per word index)")
print(f"→ {embedding_matrix.shape[1]} columns (embedding dimensions)")

print("\n" + "-" * 60)
print("Index | Word        | Embedding Vector (4 dimensions)")
print("-" * 60)

# Create reverse mapping: index → word
index_to_word = {v: k for k, v in word_index.items()}
index_to_word[0] = "<PAD>"  # Index 0 is padding

for idx in range(vocab_size):
    word = index_to_word.get(idx, "<UNK>")
    vector = embedding_matrix[idx]
    vector_str = "[" + ", ".join([f"{v:+.4f}" for v in vector]) + "]"
    print(f"  {idx:2d}  | {word:11s} | {vector_str}")

EMBEDDING MATRIX (Random Initialization)

Shape: (9, 4)
→ 9 rows (one per word index)
→ 4 columns (embedding dimensions)

------------------------------------------------------------
Index | Word        | Embedding Vector (4 dimensions)
------------------------------------------------------------
   0  | <PAD>       | [+0.0105, -0.0085, +0.0189, -0.0492]
   1  | learning    | [-0.0047, +0.0439, +0.0016, +0.0253]
   2  | i           | [-0.0342, +0.0124, +0.0092, +0.0215]
   3  | love        | [+0.0411, +0.0197, +0.0140, +0.0303]
   4  | machine     | [+0.0282, +0.0307, -0.0310, +0.0138]
   5  | deep        | [+0.0151, -0.0329, -0.0486, -0.0261]
   6  | is          | [+0.0322, -0.0377, +0.0178, +0.0313]
   7  | fun         | [-0.0368, -0.0044, +0.0157, -0.0004]
   8  | amazing     | [+0.0419, +0.0055, +0.0307, +0.0266]


## Step 8: How Embedding Lookup Works

When we pass word indices through the embedding layer, it simply **looks up** the corresponding row from the embedding matrix. This is like indexing into a dictionary!

In [8]:
# Let's trace through one sentence manually
sample_sentence = dummy_sentences[0]  # "I love machine learning"
sample_sequence = sequences[0]         # [3, 1, 4, 2]

print("=" * 60)
print("EMBEDDING LOOKUP DEMONSTRATION")
print("=" * 60)
print(f"\nSentence: '{sample_sentence}'")
print(f"Sequence: {sample_sequence}")

print("\n" + "-" * 60)
print("STEP-BY-STEP LOOKUP:")
print("-" * 60)

for word, idx in zip(sample_sentence.lower().split(), sample_sequence):
    embedding_vector = embedding_matrix[idx]
    print(f"\n  Word: '{word}'")
    print(f"  Index: {idx}")
    print(f"  Lookup: embedding_matrix[{idx}]")
    print(f"  Result: {embedding_vector.round(4)}")

EMBEDDING LOOKUP DEMONSTRATION

Sentence: 'I love machine learning'
Sequence: [2, 3, 4, 1]

------------------------------------------------------------
STEP-BY-STEP LOOKUP:
------------------------------------------------------------

  Word: 'i'
  Index: 2
  Lookup: embedding_matrix[2]
  Result: [-0.0342  0.0124  0.0092  0.0215]

  Word: 'love'
  Index: 3
  Lookup: embedding_matrix[3]
  Result: [0.0411 0.0197 0.014  0.0303]

  Word: 'machine'
  Index: 4
  Lookup: embedding_matrix[4]
  Result: [ 0.0282  0.0307 -0.031   0.0138]

  Word: 'learning'
  Index: 1
  Lookup: embedding_matrix[1]
  Result: [-0.0047  0.0439  0.0016  0.0253]


## Step 9: Output Shape of Embedding Layer

The embedding layer transforms:
- Input: `(batch_size, sequence_length)` → integers
- Output: `(batch_size, sequence_length, embedding_dim)` → dense vectors

In [9]:
# Pass all padded sequences through the embedding layer
embedded_sequences = embedding_layer(padded_sequences)

print("=" * 60)
print("EMBEDDING OUTPUT SHAPE")
print("=" * 60)
print(f"\nInput shape:  {padded_sequences.shape}")
print(f"  → {padded_sequences.shape[0]} sentences")
print(f"  → {padded_sequences.shape[1]} words per sentence (after padding)")

print(f"\nOutput shape: {embedded_sequences.shape}")
print(f"  → {embedded_sequences.shape[0]} sentences")
print(f"  → {embedded_sequences.shape[1]} words per sentence")
print(f"  → {embedded_sequences.shape[2]} dimensions per word embedding")

print("\n" + "-" * 60)
print("TRANSFORMATION:")
print("-" * 60)
print(f"\n  Each word index (scalar) → {embedding_dim}-dimensional vector")
print(f"  Each sentence ({max_length} indices) → ({max_length} × {embedding_dim}) matrix")

EMBEDDING OUTPUT SHAPE

Input shape:  (4, 5)
  → 4 sentences
  → 5 words per sentence (after padding)

Output shape: (4, 5, 4)
  → 4 sentences
  → 5 words per sentence
  → 4 dimensions per word embedding

------------------------------------------------------------
TRANSFORMATION:
------------------------------------------------------------

  Each word index (scalar) → 4-dimensional vector
  Each sentence (5 indices) → (5 × 4) matrix


In [10]:
# Visualize the embedded output for first sentence
print("=" * 60)
print("EMBEDDED SENTENCE VISUALIZATION")
print("=" * 60)
print(f"\nSentence: '{dummy_sentences[0]}'")
print(f"Padded sequence: {list(padded_sequences[0])}")

print("\nEmbedded representation (each row = one word's embedding):")
print("-" * 60)

padded_words = dummy_sentences[0].lower().split() + ["<PAD>"]  # Add padding word
for i, (word_idx, emb) in enumerate(zip(padded_sequences[0], embedded_sequences[0].numpy())):
    word = index_to_word.get(word_idx, "<PAD>")
    emb_str = "[" + ", ".join([f"{v:+.4f}" for v in emb]) + "]"
    print(f"  Position {i}: '{word:11s}' (idx={word_idx}) → {emb_str}")

EMBEDDED SENTENCE VISUALIZATION

Sentence: 'I love machine learning'
Padded sequence: [np.int32(2), np.int32(3), np.int32(4), np.int32(1), np.int32(0)]

Embedded representation (each row = one word's embedding):
------------------------------------------------------------
  Position 0: 'i          ' (idx=2) → [-0.0342, +0.0124, +0.0092, +0.0215]
  Position 1: 'love       ' (idx=3) → [+0.0411, +0.0197, +0.0140, +0.0303]
  Position 2: 'machine    ' (idx=4) → [+0.0282, +0.0307, -0.0310, +0.0138]
  Position 3: 'learning   ' (idx=1) → [-0.0047, +0.0439, +0.0016, +0.0253]
  Position 4: '<PAD>      ' (idx=0) → [+0.0105, -0.0085, +0.0189, -0.0492]


## Step 10: Building a Complete Model with Embeddings

Here's how embeddings fit into a real NLP model:

In [ ]:
from tensorflow.keras.layers import Dense, GlobalAveragePooling1D, Flatten

# Build a simple sentiment classification model
model = Sequential([
    # Layer 1: Embedding - converts word indices to dense vectors
    Embedding(input_dim=vocab_size, 
              output_dim=embedding_dim, 
              input_length=max_length,
              name='embedding_layer'),
    
    # Layer 2: Global Average Pooling - average all word embeddings
    GlobalAveragePooling1D(name='pooling_layer'),
    
    # Layer 3: Dense layer for classification
    Dense(8, activation='relu', name='hidden_layer'),
    
    # Layer 4: Output layer (binary classification)
    Dense(1, activation='sigmoid', name='output_layer')
])

model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pooling_layer                   │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ hidden_layer (Dense)            │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


MODEL EXPLANATION:

1. EMBEDDING LAYER:
   - Input: (batch, 5) - sequences of 5 word indices
   - Output: (batch, 5, 4) - 5 words × 4-dim embeddings
   - Parameters: 10 × 4 = 40 (one 4-dim vector per word)

2. GLOBAL AVERAGE POOLING:
   - Averages across all word embeddings
   - Input: (batch, 5, 4) → Output: (batch, 4)
   - Creates a fixed-size sentence representation

3. HIDDEN LAYER:
   - Dense layer with 8 neurons
   - Learns patterns from the pooled embeddings

4. OUTPUT LAYER:
   - Single neuron with sigmoid for binary classification



In [ ]:
import numpy as np

# Create a visual representation of shapes at each layer
print("\n" + "="*80)
print("DETAILED SHAPE VISUALIZATION AT EACH STEP")
print("="*80)

# Create a sample input for tracing
sample_input = np.array([[1, 2, 3, 4, 5]])  # batch_size=1, sequence_length=5

current_shape = sample_input.shape
print(f"\n{'STEP':<5} {'LAYER':<25} {'INPUT SHAPE':<20} {'OUTPUT SHAPE':<20} {'PARAMS':<10}")
print("-"*80)

# Step through each layer
for i, layer in enumerate(model.layers):
    input_shape = current_shape
    
    # Get output by passing through layer
    if i == 0:
        output = layer(sample_input)
    else:
        output = layer(output)
    
    output_shape = tuple(output.shape)
    current_shape = output_shape
    
    print(f"{i+1:<5} {layer.name:<25} {str(input_shape):<20} {str(output_shape):<20} {layer.count_params():<10}")

print("\n" + "="*80)
print("SHAPE TRANSFORMATION FLOW:")
print("="*80)

# Visual flow with descriptions
shapes_flow = [
    {'layer': 'Input', 'shape': (None, 5), 'description': 'Batch of 5 word indices'},
    {'layer': 'Embedding', 'shape': (None, 5, 4), 'description': '5 words × 4-dim vectors'},
    {'layer': 'GlobalAvgPooling1D', 'shape': (None, 4), 'description': 'Average of all word vectors'},
    {'layer': 'Dense (8 units)', 'shape': (None, 8), 'description': 'Hidden layer features'},
    {'layer': 'Dense (1 unit)', 'shape': (None, 1), 'description': 'Binary output (0 or 1)'}
]

for i, item in enumerate(shapes_flow):
    print(f"\n{item['layer']:>25}: {str(item['shape']):<15} → {item['description']}")
    if i < len(shapes_flow) - 1:
        print(" " * 25 + "↓")




DETAILED SHAPE VISUALIZATION AT EACH STEP

STEP  LAYER                     INPUT SHAPE          OUTPUT SHAPE         PARAMS    
--------------------------------------------------------------------------------
1     embedding_layer           (1, 5)               (1, 5, 4)            36        
2     pooling_layer             (1, 5, 4)            (1, 4)               0         
3     hidden_layer              (1, 4)               (1, 8)               40        
4     output_layer              (1, 8)               (1, 1)               9         

SHAPE TRANSFORMATION FLOW:

                    Input: (None, 5)       → Batch of 5 word indices
                         ↓

                Embedding: (None, 5, 4)    → 5 words × 4-dim vectors
                         ↓

       GlobalAvgPooling1D: (None, 4)       → Average of all word vectors
                         ↓

          Dense (8 units): (None, 8)       → Hidden layer features
                         ↓

           Dense (1 unit): (Non